In [ ]:
main_kaicheng.ipynb

In [ ]:
# ============================================================
# Compare five single neighbourhood-based features
# Each experiment adds only one additional feature
# to the baseline model for comparison
# ============================================================


def run_extra_feature(feature_name, f_train, f_val, f_test):
    # ------------------------------------------------------------
    # 1) Copy the original raw feature sets
    #    This prevents modification of the original data
    # ------------------------------------------------------------
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()
    Xte = X_test_raw.copy()

    # ------------------------------------------------------------
    # 2) Add the new neighbourhood-based feature
    # ------------------------------------------------------------
    Xtr[feature_name] = f_train
    Xva[feature_name] = f_val
    Xte[feature_name] = f_test

    # ------------------------------------------------------------
    # 3) Re-standardise the dataset after adding the new feature
    #    The scaler is fitted only on the training set
    #    to avoid data leakage
    # ------------------------------------------------------------
    sc = StandardScaler()

    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)
    Xte_s = sc.transform(Xte)

    # ------------------------------------------------------------
    # 4) Train a neural network model
    #    Use the same architecture as the baseline model
    #    to ensure a fair comparison
    # ------------------------------------------------------------
    

    baseline_model.fit(Xtr_s, y_train)

    # ------------------------------------------------------------
    # 5) Evaluate the model on validation and test sets
    # ------------------------------------------------------------
    baseline_val_pred = baseline_model.predict(Xva_s)
    baseline_test_pred = baseline_model.predict(Xte_s)

    val_mse = mean_squared_error(y_val, baseline_val_pred)
    test_mse = mean_squared_error(y_test, baseline_test_pred)
    return val_mse, test_mse


# ------------------------------------------------------------
# Run experiments for each neighbourhood feature separately
# ------------------------------------------------------------
single_feature_results = []

# 1) Mean price of K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceMean",
    neigh_price_mean_train, neigh_price_mean_val, neigh_price_mean_test
)
single_feature_results.append(("NeighbourPriceMean", val_mse, test_mse))


# 2) Median price of K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceMedian",
    neigh_price_median_train, neigh_price_median_val, neigh_price_median_test
)
single_feature_results.append(("NeighbourPriceMedian", val_mse, test_mse))


# 3) Minimum price among K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceMin",
    neigh_price_min_train, neigh_price_min_val, neigh_price_min_test
)
single_feature_results.append(("NeighbourPriceMin", val_mse, test_mse))


# 4) Maximum price among K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceMax",
    neigh_price_max_train, neigh_price_max_val, neigh_price_max_test
)
single_feature_results.append(("NeighbourPriceMax", val_mse, test_mse))


# 5) Price range among K nearest neighbours
val_mse, test_mse = run_extra_feature(
    "NeighbourPriceRange",
    neigh_price_range_train, neigh_price_range_val, neigh_price_range_test
)
single_feature_results.append(("NeighbourPriceRange", val_mse, test_mse))


# ------------------------------------------------------------
# Convert results into a DataFrame for easy comparison
# and sort by validation MSE
# ------------------------------------------------------------
single_feature_results_df = pd.DataFrame(single_feature_results, columns=["Feature", "Val MSE", "Test MSE"])
single_feature_results_df = single_feature_results_df.sort_values("Val MSE").reset_index(drop=True)

print(single_feature_results_df)

In [ ]:
# ============================================================
# Generate results_k_feature_df for Figure 2 (Validation MSE)
# All 2-feature combinations
# ============================================================

from sklearn.neighbors import NearestNeighbors
from itertools import combinations

k_values = [3, 5, 7, 10, 15]

def compute_features_for_k(K):
    knn = NearestNeighbors(n_neighbors=K + 1)
    knn.fit(X_train_raw[['Latitude', 'Longitude']])

    y_train_aligned = y_train.loc[X_train_raw.index].to_numpy()

    # Compute neighbour indices once
    _, idx_train = knn.kneighbors(X_train_raw[['Latitude', 'Longitude']])
    _, idx_val = knn.kneighbors(X_val_raw[['Latitude', 'Longitude']])

    # Remove self for training set
    idx_train = idx_train[:, 1:K+1]
    idx_val = idx_val[:, :K]

    # Convert neighbour indices to price matrices
    train_prices = y_train_aligned[idx_train]
    val_prices = y_train_aligned[idx_val]

    # Vectorised summaries
    neigh_price_train_k = train_prices.mean(axis=1)
    neigh_price_val_k   = val_prices.mean(axis=1)

    neigh_price_median_train_k = np.median(train_prices, axis=1)
    neigh_price_median_val_k   = np.median(val_prices, axis=1)

    neigh_price_min_train_k = train_prices.min(axis=1)
    neigh_price_min_val_k   = val_prices.min(axis=1)

    neigh_price_max_train_k = train_prices.max(axis=1)
    neigh_price_max_val_k   = val_prices.max(axis=1)

    neigh_price_range_train_k = neigh_price_max_train_k - neigh_price_min_train_k
    neigh_price_range_val_k   = neigh_price_max_val_k - neigh_price_min_val_k

    return {
        "Mean":   (neigh_price_train_k, neigh_price_val_k),
        "Median": (neigh_price_median_train_k, neigh_price_median_val_k),
        "Min":    (neigh_price_min_train_k, neigh_price_min_val_k),
        "Max":    (neigh_price_max_train_k, neigh_price_max_val_k),
        "Range":  (neigh_price_range_train_k, neigh_price_range_val_k),
    }

def run_two_features(feature_names, f_train_dict, f_val_dict):
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()

    for name in feature_names:
        Xtr[name] = f_train_dict[name]
        Xva[name] = f_val_dict[name]

    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)

    baseline_model.fit(Xtr_s, y_train)
    baseline_val_pred = baseline_model.predict(Xva_s)

    return mean_squared_error(y_val, baseline_val_pred)

k_feature_results_df = []

for K in k_values:
    feature_data = compute_features_for_k(K)
    feature_names = list(feature_data.keys())

    for combo in combinations(feature_names, 2):
        f_train_dict = {name: feature_data[name][0] for name in combo}
        f_val_dict   = {name: feature_data[name][1] for name in combo}

        val_mse = run_two_features(combo, f_train_dict, f_val_dict)

        k_feature_results_df.append({
            "k": K,
            "combo": " + ".join(combo),
            "val_mse": val_mse
        })

k_feature_results_df = pd.DataFrame(k_feature_results_df)

In [ ]:
# ============================================================
# Figure 2: Validation MSE vs Number of Neighbours (k)
# All 2-feature combinations
# ============================================================

plt.figure(figsize=(10, 6))


# ------------------------------------------------------------
# Plot validation MSE curves for each feature combination
# ------------------------------------------------------------
for combo_name in k_feature_results_df["combo"].unique():

    # Select rows corresponding to the current feature combination
    subset = k_feature_results_df[
        k_feature_results_df["combo"] == combo_name].sort_values("k")   

    # Plot validation MSE as a function of k
    plt.plot(
        subset["k"],        
        subset["val_mse"],  
        marker="o",
        label=combo_name
    )


# ------------------------------------------------------------
# Plot baseline model performance as a reference line
# ------------------------------------------------------------

plt.axhline(
    y=baseline_val_mse,
    linestyle="--",
    label="Baseline model"
)


# ------------------------------------------------------------
# Figure formatting
# ------------------------------------------------------------

plt.xlabel("Number of Neighbours (k)")
plt.ylabel("Validation MSE")


plt.title("Validation MSE vs Number of Neighbours (Two-Feature Combinations)")

# Ensure all tested k values appear on the x-axis
plt.xticks(sorted(k_feature_results_df["k"].unique()))

# Place legend outside the plot to avoid overlapping
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

# Add grid for readability
plt.grid(True, alpha=0.3)

# Display the figure
plt.show()

In [ ]:
# ============================================================
# Final comparison
# ============================================================
# ============================================================
# Model 2 Train MSE
# ============================================================

train_pred_enh_final = model_enh_final.predict(X_train_enh_final)

enhanced_train_mse_final = mean_squared_error(
    y_train_final,
    train_pred_enh_final
)

print("\nFinal Enhanced Model Performance")
print("Final Enhanced Train MSE:", enhanced_train_mse_final)
print("Final Enhanced Test MSE:", enhanced_test_mse_final)
print("Enhanced Gap (Test - Train):", enhanced_test_mse_final - enhanced_train_mse_final)


print("\n================ FINAL MODEL COMPARISON ================")

print("Baseline Train MSE:", baseline_train_mse)
print("Baseline Test MSE:", baseline_test_mse_final)
print("Baseline Gap (Test - Train):", baseline_test_mse_final - baseline_train_mse)

print("\nEnhanced Train MSE:", enhanced_train_mse_final)
print("Enhanced Test MSE:", enhanced_test_mse_final)
print("Enhanced Gap (Test - Train):", enhanced_test_mse_final - enhanced_train_mse_final)

print("\nTest MSE Improvement (Enhanced - Baseline):",
      enhanced_test_mse_final - baseline_test_mse_final)